# 🌿 Brouillon — Decision Tree Regressor

> **Statut** : ❌ **Modèle écarté** — Performances insuffisantes (R² = 0.762, CV RMSE = 0.201)  
> Ce notebook documente les expérimentations réalisées avec l'arbre de décision.  
> **Conclusion** : overfitting marqué, performances nettement inférieures aux modèles ensemblistes.

---
**Projet** : House Price Prediction — Ames Iowa  
**Auteur** : [Votre Nom]  
**Date** : Avril 2026

## 1. Imports & Chargement des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os

from sklearn.tree import DecisionTreeRegressor, export_text, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

sys.path.append('../../src')
from preprocessing import full_pipeline

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Blues_r')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

train = pd.read_csv('../../data/train.csv')
test  = pd.read_csv('../../data/test.csv')
print(f'Train : {train.shape} | Test : {test.shape}')

## 2. Preprocessing

In [ ]:
X, X_test, y, scaler = full_pipeline(train, test)
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f'Features : {X.shape[1]} | Train : {X_tr.shape[0]} | Val : {X_val.shape[0]}')

## 3. Essai 1 — Arbre sans contrainte (overfitting)

In [ ]:
# Arbre sans contrainte → parfait sur train, mauvais sur test
dt_unconstrained = DecisionTreeRegressor(random_state=RANDOM_STATE)
dt_unconstrained.fit(X_tr, y_tr)

train_r2 = r2_score(y_tr, dt_unconstrained.predict(X_tr))
val_r2   = r2_score(y_val, dt_unconstrained.predict(X_val))

print(f'=== Arbre sans contrainte ===')
print(f'R² Train  : {train_r2:.4f}  ← overfitting total')
print(f'R² Val    : {val_r2:.4f}   ← mauvais en généralisation')

## 4. Essai 2 — Contrôle de la profondeur (max_depth)

In [ ]:
depths   = range(2, 15)
tr_r2s, val_r2s = [], []

for d in depths:
    dt = DecisionTreeRegressor(max_depth=d, random_state=RANDOM_STATE)
    dt.fit(X_tr, y_tr)
    tr_r2s.append(r2_score(y_tr, dt.predict(X_tr)))
    val_r2s.append(r2_score(y_val, dt.predict(X_val)))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(depths, tr_r2s,  'o-', color='#2563EB', label='Train R²')
ax.plot(depths, val_r2s, 's--', color='#EF4444', label='Val R²')
ax.axvline(6, color='gray', linestyle=':', alpha=.7, label='Profondeur choisie=6')
ax.set_xlabel('max_depth')
ax.set_ylabel('R²')
ax.set_title('Compromis Biais / Variance — Decision Tree', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../../data/dtree_analysis.png', dpi=120)
plt.show()

best_depth = list(depths)[val_r2s.index(max(val_r2s))]
print(f'Meilleure profondeur : {best_depth} | Val R² = {max(val_r2s):.4f}')

## 5. Essai 3 — GridSearchCV sur hyperparamètres

In [ ]:
param_grid = {
    'max_depth':        [4, 5, 6, 7, 8],
    'min_samples_split':[5, 10, 20, 30],
    'min_samples_leaf': [2, 5, 10, 15],
    'max_features':     ['sqrt', 'log2', None],
}
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

gs = GridSearchCV(
    DecisionTreeRegressor(random_state=RANDOM_STATE),
    param_grid, cv=kf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1, verbose=0
)
gs.fit(X_tr, y_tr)

best_dt = gs.best_estimator_
cv_rmse = -gs.best_score_
val_r2  = r2_score(y_val, best_dt.predict(X_val))

print('=== GridSearch Result ===')
print('Meilleurs paramètres :', gs.best_params_)
print(f'CV RMSE : {cv_rmse:.4f}')
print(f'Val R²  : {val_r2:.4f}')

## 6. Analyse des erreurs — Résidus

In [ ]:
preds    = best_dt.predict(X_val)
residuals = y_val - preds

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(preds, residuals, alpha=.5, color='#2563EB', s=20)
axes[0].axhline(0, color='#EF4444', lw=1.5)
axes[0].set_xlabel('Prédictions')
axes[0].set_ylabel('Résidus')
axes[0].set_title('Résidus — Decision Tree', fontweight='bold')

axes[1].hist(residuals, bins=40, color='#2563EB', alpha=.75, edgecolor='white')
axes[1].set_xlabel('Résidu')
axes[1].set_title('Distribution des Résidus', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'RMSE  : {np.sqrt(mean_squared_error(y_val, preds)):.4f}')
print(f'MAE   : {mean_absolute_error(y_val, preds):.4f}')
print(f'R²    : {r2_score(y_val, preds):.4f}')

## 7. Feature Importance

In [ ]:
import pandas as pd

fi = pd.Series(best_dt.feature_importances_, index=X.columns)
top15 = fi.nlargest(15)

fig, ax = plt.subplots(figsize=(8, 5))
top15.plot.barh(ax=ax, color='#2563EB', alpha=.8)
ax.invert_yaxis()
ax.set_title('Top 15 Feature Importances — Decision Tree', fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 8. ❌ Conclusion — Pourquoi ce modèle est écarté

| Métrique | Decision Tree | **Gradient Boosting** | **Random Forest** |
|----------|-------------|---------------------|------------------|
| CV RMSE  | 0.201       | **0.118**           | 0.138            |
| Val R²   | 0.762       | **0.905**           | 0.872            |

**Raisons d'élimination :**
1. **Overfitting sévère** — L'arbre mémorise les données d'entraînement (R² train ≈ 1.0, R² val ≈ 0.76)
2. **CV RMSE = 0.201** — le plus mauvais de tous les modèles testés
3. **Instabilité** — Très sensible aux perturbations dans les données  
4. **Pas de régularisation naturelle** — Contrairement à Random Forest (bagging) ou Gradient Boosting

> **Action** : ❌ Modèle écarté. Non retenu pour le notebook final.